In [ ]:
"""
=========================================================
PHISHING DATASET BUILDER (FINAL ROBUST VERSION)
=========================================================
- Pulls phishing URLs from:
    • URLHaus
    • OpenPhish
    • PhishTank
- Pulls legitimate domains from:
    • Majestic Million
- Automatically adjusts if phishing pool is smaller
- Augments legitimate login URLs
- Cleans & balances dataset
- Saves final CSV locally
=========================================================
"""

import pandas as pd
import requests
from io import StringIO

# ==============================
# CONFIGURATION
# ==============================

TARGET_TOTAL = 200000
PHISH_RATIO = 0.4   # 40% phishing / 60% legit

NUM_PHISH = int(TARGET_TOTAL * PHISH_RATIO)
NUM_LEGIT = TARGET_TOTAL - NUM_PHISH

OUTPUT_FILE = "phishing_dataset_clean_final.csv"

print("\n===== BUILDING CLEAN REALISTIC DATASET =====\n")

# ==============================
# SAFE REQUEST FUNCTION
# ==============================

def safe_request(url, name):
    try:
        print(f"Downloading {name}...")
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        print(f"{name} download successful.")
        return response
    except Exception as e:
        print(f"[WARNING] {name} failed:", e)
        return None


# ==============================
# PHISHING SOURCES
# ==============================

def download_urlhaus():
    url = "https://urlhaus.abuse.ch/downloads/csv_recent/"
    response = safe_request(url, "URLHaus")
    if response is None:
        return pd.DataFrame(columns=["url"])
    df = pd.read_csv(StringIO(response.text), comment="#", header=None)
    df = df[[2]]
    df.columns = ["url"]
    return df


def download_openphish():
    url = "https://openphish.com/feed.txt"
    response = safe_request(url, "OpenPhish")
    if response is None:
        return pd.DataFrame(columns=["url"])
    lines = response.text.splitlines()
    return pd.DataFrame(lines, columns=["url"])


def download_phishtank():
    try:
        print("Downloading PhishTank...")
        url = "https://data.phishtank.com/data/online-valid.csv"
        df = pd.read_csv(url)
        df = df[["url"]]
        print("PhishTank download successful.")
        return df
    except Exception as e:
        print("[WARNING] PhishTank failed:", e)
        return pd.DataFrame(columns=["url"])


# ==============================
# LEGITIMATE SOURCE
# ==============================

def download_majestic():
    try:
        print("Downloading Majestic Million...")
        url = "https://downloads.majestic.com/majestic_million.csv"
        df = pd.read_csv(url)
        domains = df["Domain"].dropna().unique()
        urls = ["https://" + d for d in domains]
        print("Majestic download successful.")
        return pd.DataFrame(urls, columns=["url"])
    except Exception as e:
        print("[WARNING] Majestic failed:", e)
        return pd.DataFrame(columns=["url"])


# ==============================
# DOWNLOAD DATA
# ==============================

df_phish_pool = pd.concat([
    download_urlhaus(),
    download_openphish(),
    download_phishtank()
], ignore_index=True)

df_phish_pool.dropna(inplace=True)
df_phish_pool.drop_duplicates(inplace=True)

print("\nTotal phishing pool:", len(df_phish_pool))

# ==============================
# AUTO ADJUST IF NEEDED
# ==============================

if len(df_phish_pool) < NUM_PHISH:
    print("\n[WARNING] Not enough phishing URLs available.")
    print("Adjusting phishing count to available pool.")

    NUM_PHISH = len(df_phish_pool)
    NUM_LEGIT = int(NUM_PHISH * ((1 - PHISH_RATIO) / PHISH_RATIO))

    print(f"Adjusted phishing count: {NUM_PHISH}")
    print(f"Adjusted legit count: {NUM_LEGIT}")

# ==============================
# DOWNLOAD LEGITIMATE DOMAINS
# ==============================

df_legit_pool = download_majestic()
df_legit_pool.drop_duplicates(inplace=True)

print("Total legit pool:", len(df_legit_pool))

if len(df_legit_pool) < NUM_LEGIT:
    print("\n[WARNING] Legit pool smaller than requested.")
    NUM_LEGIT = len(df_legit_pool)
    print(f"Adjusted legit count: {NUM_LEGIT}")

# ==============================
# SAMPLE & BALANCE
# ==============================

df_phish = df_phish_pool.sample(n=NUM_PHISH, random_state=42)
df_phish["label"] = 1

df_legit = df_legit_pool.sample(n=NUM_LEGIT, random_state=42)
df_legit["label"] = 0

# ==============================
# LEGITIMATE LOGIN AUGMENTATION
# ==============================

print("Adding realistic legitimate login URLs...")

trusted_login_domains = [
    "google.com",
    "github.com",
    "microsoftonline.com",
    "amazon.in",
    "paypal.com",
    "apple.com",
    "facebook.com"
]

login_paths = [
    "/login",
    "/signin",
    "/account",
    "/auth",
    "/secure",
    "/ap/signin"
]

augmented_urls = []

for domain in trusted_login_domains:
    for path in login_paths:
        augmented_urls.append("https://" + domain + path)
        augmented_urls.append("https://accounts." + domain)
        augmented_urls.append("https://login." + domain)

df_augmented_legit = pd.DataFrame({
    "url": augmented_urls,
    "label": 0
})

df_legit = pd.concat([df_legit, df_augmented_legit], ignore_index=True)
df_legit.drop_duplicates(subset="url", inplace=True)

print("Augmented legit size:", len(df_legit))

# ==============================
# FINAL CLEANING
# ==============================

df_final = pd.concat([df_legit, df_phish], ignore_index=True)

df_final.drop_duplicates(subset="url", inplace=True)
df_final = df_final[df_final["url"].str.startswith("http")]

df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nFinal distribution:")
print(df_final["label"].value_counts())

print("\nTotal size:", df_final.shape)

# ==============================
# SAVE
# ==============================

df_final.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ DATASET SAVED AS: {OUTPUT_FILE}")
print("\nBuild complete.\n")



===== BUILDING CLEAN REALISTIC DATASET =====

URLHaus download successful.
OpenPhish download successful.
PhishTank download successful.

Total phishing pool: 74470

[WARNING] Not enough phishing URLs available.
Adjusting phishing count to available pool.
Adjusted phishing count: 74470
Adjusted legit count: 111704
Majestic download successful.
Total legit pool: 1000000
Adding realistic legitimate login URLs...
Augmented legit size: 111760

Final distribution:
label
0    111760
1     74470
Name: count, dtype: int64

Total size: (186230, 2)

✅ DATASET SAVED AS: phishing_dataset_clean_final.csv

Build complete.

